# 1.2 Data for the Survival Analysis

**For Class 5 (3.4 survival_analysis):** pull what we need to study **time from a diabetes diagnosis to chronic kidney disease (CKD)**, for the people in our population **with diabetes**.

| Query | What | Saved as |
|---|---|---|
| Conditions | every diabetes and CKD diagnosis, with its date | `survival_conditions.csv` |
| Visits | every visit, reduced to each person's **last** visit date | `last_visits.csv` |

Like 1.1, these queries were **generated by the classic Cohort Builder**; the only change is the dataset path, `` `{CURRENT_DATASET}.table` ``.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

## Query 1: Diabetes and CKD diagnoses

In [ ]:
condition_sql <- r"---(
    SELECT
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `{CURRENT_DATASET}.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `{CURRENT_DATASET}.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` cr       
                    WHERE
                        concept_id IN (201820, 37017104, 37017813, 37018761, 43531559, 43531562, 43531566, 43531577, 43531578, 43531653, 443597, 443601, 443611, 443612, 443614, 44782429, 44782689, 44782690, 44782691, 44782692, 45763854, 45763855, 45769901, 45769903, 45771075, 45773576, 45773688, 46270353, 46271022, 46273636)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `{CURRENT_DATASET}.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_person` p 
                    WHERE
                        DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                        AND NOT EXISTS (      SELECT
                            'x'      
                        FROM
                            `{CURRENT_DATASET}.death` d      
                        WHERE
                            d.person_id = p.person_id ) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        ethnicity_concept_id IN (38003564, 38003563) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        gender_concept_id IN (45878463, 45880669) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        race_concept_id IN (8527, 8516, 8515) ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `{CURRENT_DATASET}.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) c_occurrence 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id
)---"

conditions <- run_query(str_glue(condition_sql))
glimpse(conditions)

Each row is one diagnosis. We flag it from the condition **name**:

- `is_diabetes`: the name mentions diabetes
- `is_ckd`: the name mentions kidney or renal disease

A diagnosis like *"CKD stage 3 due to type 2 diabetes"* counts as **both**.

In [ ]:
survival_conditions <- conditions %>%
  transmute(
    person_id,
    condition_start_date = as.Date(condition_start_datetime),
    condition_name       = standard_concept_name,
    is_diabetes = str_detect(str_to_lower(condition_name), "diabet"),
    is_ckd      = str_detect(str_to_lower(condition_name), "kidney|renal")
  )

survival_conditions %>% count(is_diabetes, is_ckd)

In [ ]:
save_data(survival_conditions, "survival_conditions.csv")

## Query 2: Visits

People who never develop CKD are **censored** at their last visit: we know they were CKD-free until then. The query returns every visit; we keep each person's latest.

In [ ]:
visit_sql <- r"---(
    SELECT
        visit.PERSON_ID,
        v_standard_concept.concept_name as standard_concept_name,
        visit.visit_start_datetime,
        visit.visit_end_datetime 
    FROM
        `{CURRENT_DATASET}.visit_occurrence` visit 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` v_standard_concept 
            ON visit.visit_concept_id = v_standard_concept.concept_id 
    WHERE
        (
            visit_concept_id IN (5083, 581477, 9201, 9202, 9203)
        )  
        AND (
            visit.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `{CURRENT_DATASET}.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                    AND NOT EXISTS (      SELECT
                        'x'      
                    FROM
                        `{CURRENT_DATASET}.death` d      
                    WHERE
                        d.person_id = p.person_id ) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    ethnicity_concept_id IN (38003564, 38003563) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    gender_concept_id IN (45878463, 45880669) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    race_concept_id IN (8527, 8516, 8515) ) 
                AND cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` cr       
                            WHERE
                                concept_id IN (201820)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )
        )
)---"

visits <- run_query(str_glue(visit_sql))
glimpse(visits)

In [ ]:
last_visits <- visits %>%
  rename_with(str_to_lower) %>%                       # PERSON_ID -> person_id
  mutate(visit_date = as.Date(coalesce(visit_end_datetime, visit_start_datetime))) %>%
  group_by(person_id) %>%
  summarise(last_visit_date = max(visit_date, na.rm = TRUE))

save_data(last_visits, "last_visits.csv")
list.files(DATA_DIR)